# Mask R-CNN Training (ArtInsight)

This notebook is meant to get a stable baseline running in Colab. The priority is simple: make the full pipeline work end to end before tuning anything.

Workflow:
1. mount Drive,
2. load VIA annotations,
3. validate masks/boxes visually,
4. train 1 epoch first.

## 1. Mount Google Drive

Colab sessions are temporary. Mounting Drive lets us keep checkpoints and outputs after runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## 2. Imports

Minimal stack for JSON parsing, mask creation, visualization, and Mask R-CNN training with `torchvision`.

In [ ]:
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import cv2
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


## 3. Runtime Check

If no GPU is available, stop here. Training Mask R-CNN on CPU is too slow for practical iteration.

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

if device.type != "cuda":
    print("WARNING: GPU not detected. Go to Runtime > Change runtime type > GPU.")


## 4. Paths

Set these paths once and keep path logic centralized in this section.

In [ ]:
ROOT = Path("/content/drive/MyDrive/tfg/artinsight")
IMG_DIR = ROOT / "images"
CHECKPOINT_DIR = ROOT / "checkpoints"
OUTPUT_DIR = ROOT / "outputs"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANNOTATIONS = {
    "lpl_train": ROOT / "lpl_train.json",
    "lpl_test": ROOT / "lpl_test.json",
    "stucco_train": ROOT / "stucco_train.json",
    "stucco_test": ROOT / "stucco_test.json",
}

print("ROOT:", ROOT)
print("IMG_DIR exists:", IMG_DIR.exists())
for k, v in ANNOTATIONS.items():
    print(f"{k}: {v} | exists={v.exists()}")


## 5. Helpers: Load JSON and Build a Unified Sample List

We merge the 4 JSON files (`lpl/stucco` x `train/test`) into one sample list so downstream code stays consistent.

In [ ]:
def load_via_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def build_samples(annotation_paths):
    samples = []

    for key, path in annotation_paths.items():
        task, split = key.split("_")
        data = load_via_json(path)

        for _, entry in data.items():
            samples.append({
                "filename": entry["filename"],
                "regions": entry.get("regions", {}),
                "task": task,
                "split": split,
            })

    return samples

samples = build_samples(ANNOTATIONS)
print("Total samples:", len(samples))


## 6. Quick Composition Check

Small but important check to confirm sample counts and train/test split look correct.

In [ ]:
split_counter = Counter((s["task"], s["split"]) for s in samples)
print("Samples by (task, split):")
for k, v in sorted(split_counter.items()):
    print(k, "->", v)

empty_region_samples = sum(1 for s in samples if len(s["regions"]) == 0)
print("\nSamples with zero raw regions:", empty_region_samples)


## 7. Polygon to Mask

Annotations come as polygon points; the model needs binary instance masks.

In [ ]:
def polygon_to_mask(height, width, all_x, all_y):
    if len(all_x) < 3 or len(all_x) != len(all_y):
        return None

    mask = np.zeros((height, width), dtype=np.uint8)
    pts = np.array(list(zip(all_x, all_y)), dtype=np.int32)
    cv2.fillPoly(mask, [pts], 1)
    return mask


## 8. Label Policy

For the first baseline we use one positive class: `damage=1` (background is `0`).
This keeps the setup stable on a small dataset.

In [ ]:
def get_label_id(task, region_attributes, single_class=True):
    if single_class:
        return 1

    # Optional multi-class mapping for later experiments
    if task == "stucco":
        return 1
    elif task == "lpl":
        return 2
    else:
        raise ValueError(f"Unknown task: {task}")


## 9. Dataset Class for Mask R-CNN

Returns `image` + `target` (`boxes`, `labels`, `masks`, etc.) and filters invalid annotations to avoid training-time crashes.

In [ ]:
class ArtInsightMaskRCNNDataset(Dataset):
    def __init__(self, samples, img_dir, single_class=True, min_area=25, resize=(512, 512)):
        self.samples = samples
        self.img_dir = Path(img_dir)
        self.single_class = single_class
        self.min_area = min_area
        self.resize = resize  # tuple (W, H) or None

    def __len__(self):
        return len(self.samples)

    def _resolve_image_path(self, filename):
        return self.img_dir / filename

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image_path = self._resolve_image_path(sample["filename"])

        img = Image.open(image_path).convert("RGB")
        img_np = np.array(img)
        orig_h, orig_w = img_np.shape[:2]

        if self.resize is not None:
            new_w, new_h = self.resize
            img_np = cv2.resize(img_np, (new_w, new_h), interpolation=cv2.INTER_AREA)
        else:
            new_h, new_w = orig_h, orig_w

        scale_x = new_w / orig_w
        scale_y = new_h / orig_h

        masks = []
        boxes = []
        labels = []

        for region in sample["regions"].values():
            shape = region.get("shape_attributes", {})
            region_attributes = region.get("region_attributes", {})

            if shape.get("name") != "polygon":
                continue

            all_x = shape.get("all_points_x", [])
            all_y = shape.get("all_points_y", [])

            if len(all_x) < 3 or len(all_x) != len(all_y):
                continue

            # Scale polygon coordinates if resize is enabled
            scaled_x = [int(round(x * scale_x)) for x in all_x]
            scaled_y = [int(round(y * scale_y)) for y in all_y]

            mask = polygon_to_mask(new_h, new_w, scaled_x, scaled_y)
            if mask is None or mask.sum() == 0:
                continue

            y_idx, x_idx = np.where(mask == 1)
            if len(x_idx) == 0 or len(y_idx) == 0:
                continue

            xmin = np.min(x_idx)
            xmax = np.max(x_idx)
            ymin = np.min(y_idx)
            ymax = np.max(y_idx)

            if xmax <= xmin or ymax <= ymin:
                continue

            area = (xmax - xmin) * (ymax - ymin)
            if area < self.min_area:
                continue

            masks.append(mask)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(get_label_id(sample["task"], region_attributes, self.single_class))

        if len(masks) > 0:
            masks = torch.as_tensor(np.stack(masks), dtype=torch.uint8)
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
        else:
            masks = torch.zeros((0, new_h, new_w), dtype=torch.uint8)
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": torch.tensor([idx], dtype=torch.int64),
            "area": area,
            "iscrowd": iscrowd,
        }

        img_tensor = torch.as_tensor(img_np / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return img_tensor, target


## 10. Build Train/Test Datasets

We keep the original split to avoid leakage and preserve comparability.

In [ ]:
train_samples = [s for s in samples if s["split"] == "train"]
test_samples = [s for s in samples if s["split"] == "test"]

train_dataset = ArtInsightMaskRCNNDataset(
    train_samples,
    IMG_DIR,
    single_class=True,
    min_area=25,
    resize=(512, 512)
)

test_dataset = ArtInsightMaskRCNNDataset(
    test_samples,
    IMG_DIR,
    single_class=True,
    min_area=25,
    resize=(512, 512)
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))


## 11. Inspect One Sample

Quick check for tensor shapes and instance counts before training.

In [ ]:
img_tensor, target = train_dataset[0]

print("Image tensor shape:", img_tensor.shape)
print("Target keys:", list(target.keys()))
print("Boxes shape:", target["boxes"].shape)
print("Masks shape:", target["masks"].shape)
print("Labels shape:", target["labels"].shape)
print("Instances in sample 0:", len(target["boxes"]))


## 12. Fast Visualization

Designed for Colab speed: enough visual detail to debug without heavy rendering overhead.

In [ ]:
def show_sample_fast(dataset, idx, show_boxes=True, alpha=0.35):
    img, target = dataset[idx]
    img_np = img.permute(1, 2, 0).numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(img_np)

    if len(target["masks"]) > 0:
        combined_mask = np.max(target["masks"].numpy(), axis=0)

        overlay = np.zeros((combined_mask.shape[0], combined_mask.shape[1], 4), dtype=np.float32)
        overlay[..., 0] = 1.0  # red
        overlay[..., 3] = combined_mask * alpha
        plt.imshow(overlay)

    if show_boxes and len(target["boxes"]) > 0:
        for box in target["boxes"].numpy():
            xmin, ymin, xmax, ymax = box
            plt.gca().add_patch(
                plt.Rectangle(
                    (xmin, ymin),
                    xmax - xmin,
                    ymax - ymin,
                    fill=False,
                    edgecolor="yellow",
                    linewidth=1
                )
            )

    plt.title(f"Sample {idx} | instances={len(target['boxes'])}")
    plt.axis("off")
    plt.show()


## 13. Visual Sanity Check

Critical step: verify masks and boxes align with the image.

In [ ]:
show_sample_fast(train_dataset, 0)
show_sample_fast(train_dataset, min(1, len(train_dataset) - 1))
show_sample_fast(test_dataset, 0)


## 14. Dataset Summary After Filtering

Shows how much data remains after cleaning rules and `min_area` filtering.

In [ ]:
def summarize_dataset(dataset):
    total_instances = 0
    empty_images = 0
    per_image = []

    for i in range(len(dataset)):
        _, target = dataset[i]
        n = len(target["boxes"])
        total_instances += n
        per_image.append(n)
        if n == 0:
            empty_images += 1

    print("Number of images:", len(dataset))
    print("Total valid instances:", total_instances)
    print("Images with zero valid instances:", empty_images)
    print("Min instances/image:", min(per_image) if per_image else 0)
    print("Max instances/image:", max(per_image) if per_image else 0)
    print("Mean instances/image:", float(np.mean(per_image)) if per_image else 0)

print("Train dataset summary")
summarize_dataset(train_dataset)

print("\nTest dataset summary")
summarize_dataset(test_dataset)


## 15. DataLoader

We use a custom `collate_fn` because each image can have a different number of instances.

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=1,   # Colab-safe starting point
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)


## 16. Build the Mask R-CNN Model

Start from pretrained weights and replace heads for `num_classes=2` (background + damage).

In [ ]:
def get_model(num_classes=2):
    model = maskrcnn_resnet50_fpn(weights="DEFAULT")

    # Replace box predictor
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Replace mask predictor
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask,
        hidden_layer,
        num_classes
    )

    return model

model = get_model(num_classes=2)
model.to(device)


## 17. Optimizer and Scheduler

Conservative defaults first. We can tune later once the baseline is stable.

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)


## 18. Smoke Test (1 Batch)

Run one batch through the model before full training to catch malformed targets early.

In [ ]:
images, targets = next(iter(train_loader))

images = [img.to(device) for img in images]
targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

model.train()
loss_dict = model(images, targets)
losses = sum(loss for loss in loss_dict.values())

print("Loss dict keys:", loss_dict.keys())
print("Total loss:", float(losses.item()))


## 19. Training Loop

Start with 1 epoch. If stable, increase to 5+ epochs.

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    running_loss = 0.0

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item()

    return running_loss / max(len(data_loader), 1)


In [ ]:
num_epochs = 1  # Start with 1. Increase only after validation.

history = []

for epoch in range(num_epochs):
    epoch_loss = train_one_epoch(model, optimizer, train_loader, device)
    history.append(epoch_loss)

    print(f"Epoch {epoch + 1}/{num_epochs} - loss: {epoch_loss:.4f}")

    lr_scheduler.step()

    checkpoint_path = CHECKPOINT_DIR / f"maskrcnn_epoch_{epoch+1}.pth"
    torch.save(model.state_dict(), checkpoint_path)
    print("Saved checkpoint to:", checkpoint_path)


## 20. Loss Curve

Use this as a quick health signal, not as final model evaluation.

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(history) + 1), history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Mask R-CNN training loss")
plt.grid(True)
plt.show()


## 21. Inference on Test Samples

After initial training, confirm the model produces plausible outputs without errors.

In [ ]:
model.eval()

images, targets = next(iter(test_loader))
images = [img.to(device) for img in images]

with torch.no_grad():
    outputs = model(images)

print(outputs[0].keys())
print("Predicted boxes:", outputs[0]["boxes"].shape)
print("Predicted masks:", outputs[0]["masks"].shape)
print("Predicted scores:", outputs[0]["scores"].shape)


## 22. Visualize Predictions

Apply a confidence threshold to avoid noisy visualizations.

In [ ]:
def show_predictions(image_tensor, output, score_threshold=0.5, alpha=0.35):
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(img_np)

    scores = output["scores"].detach().cpu().numpy()
    boxes = output["boxes"].detach().cpu().numpy()
    masks = output["masks"].detach().cpu().numpy()

    keep = scores >= score_threshold

    if keep.sum() > 0:
        kept_masks = masks[keep, 0]
        combined_mask = np.max((kept_masks > 0.5).astype(np.uint8), axis=0)

        overlay = np.zeros((combined_mask.shape[0], combined_mask.shape[1], 4), dtype=np.float32)
        overlay[..., 1] = 1.0  # green
        overlay[..., 3] = combined_mask * alpha
        plt.imshow(overlay)

        for box in boxes[keep]:
            xmin, ymin, xmax, ymax = box
            plt.gca().add_patch(
                plt.Rectangle(
                    (xmin, ymin),
                    xmax - xmin,
                    ymax - ymin,
                    fill=False,
                    edgecolor="cyan",
                    linewidth=1
                )
            )

    plt.title(f"Predictions above threshold={score_threshold}")
    plt.axis("off")
    plt.show()


In [ ]:
show_predictions(images[0].cpu(), outputs[0], score_threshold=0.5)


## 23. Save One Prediction

Useful for reports and for keeping qualitative results outside temporary Colab sessions.

In [ ]:
def save_prediction_plot(image_tensor, output, output_path, score_threshold=0.5, alpha=0.35):
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(img_np)

    scores = output["scores"].detach().cpu().numpy()
    boxes = output["boxes"].detach().cpu().numpy()
    masks = output["masks"].detach().cpu().numpy()

    keep = scores >= score_threshold

    if keep.sum() > 0:
        kept_masks = masks[keep, 0]
        combined_mask = np.max((kept_masks > 0.5).astype(np.uint8), axis=0)

        overlay = np.zeros((combined_mask.shape[0], combined_mask.shape[1], 4), dtype=np.float32)
        overlay[..., 1] = 1.0
        overlay[..., 3] = combined_mask * alpha
        plt.imshow(overlay)

        for box in boxes[keep]:
            xmin, ymin, xmax, ymax = box
            plt.gca().add_patch(
                plt.Rectangle(
                    (xmin, ymin),
                    xmax - xmin,
                    ymax - ymin,
                    fill=False,
                    edgecolor="cyan",
                    linewidth=1
                )
            )

    plt.title(f"Predictions above threshold={score_threshold}")
    plt.axis("off")
    plt.savefig(output_path, bbox_inches="tight")
    plt.show()
    plt.close()

save_prediction_plot(
    images[0].cpu(),
    outputs[0],
    OUTPUT_DIR / "sample_prediction.png",
    score_threshold=0.5
)

print("Saved:", OUTPUT_DIR / "sample_prediction.png")


## 24. Next Steps

Recommended order:
1. move from 1 to 5 epochs,
2. review qualitative errors,
3. tune `min_area` and `score_threshold`,
4. only then try a multi-class variant.